## import data set needed
 - data set for questions = queries
 - data set for answers = text
 - data set for QAR = qrels

In [1]:
from datasets import load_dataset

dataset_answers = load_dataset("BeIR/scifact", "corpus", split="corpus" , trust_remote_code=True)
dataset_answers[0]

d:\anaconda\envs\qdrant\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'_id': '4983',
 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.',
 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, t

In [2]:
dataset_questions = load_dataset("BeIR/scifact", "queries", split="queries" , trust_remote_code=True)
dataset_questions[0]

d:\anaconda\envs\qdrant\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\basil\.cache\huggingface\hub\datasets--BeIR--scifact. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating queries split: 100%|██████████| 1109/1109 [00:00<00:00, 221594.17 examples/s]


{'_id': '0',
 'title': '',
 'text': '0-dimensional biomaterials lack inductive properties.'}

In [17]:
qrels = load_dataset("BeIR/scifact-qrels", split="train" , trust_remote_code=True)
qrels[0]

{'query-id': 0, 'corpus-id': 31715818, 'score': 1}

In [18]:
from collections import defaultdict

class CustomQrels:
    def __init__(self, qrels_dict, name=None):
        self.qrels = qrels_dict
        self.name = name
        
    def get_query_ids(self):
        """Return all query IDs in the qrels."""
        return list(self.qrels.keys())
    
    def get_document_ids(self, query_id):
        """Return all document IDs for a given query ID."""
        return list(self.qrels.get(query_id, {}).keys())
    
    def get_relevance(self, query_id, doc_id):
        """Return the relevance score for a query-document pair."""
        return self.qrels.get(query_id, {}).get(doc_id, 0)
    
    def __str__(self):
        return f"CustomQrels(name={self.name}, queries={len(self.get_query_ids())})"

## transform qrels to dictionary inside a dictionary


In [19]:
from collections import defaultdict

qrels_dict = defaultdict(dict)
if not isinstance(qrels, CustomQrels):
    for entry in qrels:
        qrels_id = str(entry['query-id'])
        doc_id = str(entry['corpus-id'])
        qrels_dict[qrels_id][doc_id] = entry['score']
custom_qrels = CustomQrels(qrels_dict , name = "scifact")

## create vector database baseline and upload embeddings of answers inside it

In [23]:
from qdrant_client import QdrantClient
client = QdrantClient(":memory:" , timeout = 600)


In [31]:
from fastembed import TextEmbedding
from fastembed.late_interaction import LateInteractionTextEmbedding
from fastembed.sparse.bm25 import Bm25
from qdrant_client import models

dense_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
late_interaction = LateInteractionTextEmbedding("colbert-ir/colbertv2.0" , device = "cuda")
bm25_embedding_model = Bm25("Qdrant/bm25")

sample_text = ["This is a sample document"]
dense_embedding = list(dense_model.passage_embed(sample_text))[0]
late_embedding = list(late_interaction.passage_embed(sample_text))[0]

In [35]:
len(late_embedding[0])

128

In [37]:
from fastembed import TextEmbedding
from fastembed.late_interaction import LateInteractionTextEmbedding
from fastembed.sparse.bm25 import Bm25
from qdrant_client import models

dense_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
late_interaction = LateInteractionTextEmbedding("colbert-ir/colbertv2.0" , device = "cuda")
bm25_embedding_model = Bm25("Qdrant/bm25")

sample_text = ["This is a sample document"]
dense_embedding = list(dense_model.passage_embed(sample_text))[0]
late_embedding = list(late_interaction.passage_embed(sample_text))[0]


client.create_collection(
    "scifact",
    vectors_config ={
        "all-MiniLM-L6-v2" : models.VectorParams(
            size = len(dense_embedding),
            distance = models.Distance.COSINE
        ),
        "colbertv2.0" : models.VectorParams(
            size = len(late_embedding[0]),
            distance = models.Distance.COSINE,
            multivector_config = models.MultiVectorConfig(
            comparator = models.MultiVectorComparator.MAX_SIM,
            ),
        ),
        
    },
    sparse_vectors_config = {
        "bm25" : models.SparseVectorParams(
            modifier= models.Modifier.IDF,
            )
    }
)

True

### vectoring ANSWER data into vectore database

In [44]:
import tqdm
batch_size = 4

for batch in tqdm.tqdm(dataset_answers.iter(batch_size=batch_size) , total = len(dataset_answers) // batch_size):
    dense_embeddings = list(dense_model.passage_embed(batch["text"]))
    late_embeddings = list(late_interaction.passage_embed(batch["text"]))
    sparse_embeddings = list(bm25_embedding_model.passage_embed(batch["text"]))

    client.upload_points(
        "scifact",
        points = [
            models.PointStruct(
                id = int(batch["_id"][i]),
                vector = {
                    "all-MiniLM-L6-v2": dense_embeddings[i].tolist(),
                    "colbertv2.0": late_embeddings[i].tolist(),
                    "bm25": models.SparseVector(
                        indices = sparse_embeddings[i].as_object()["indices"].tolist(),
                        values = sparse_embeddings[i].as_object()["values"].tolist(),
                    ),
                },
                payload = {
                    "_id": batch["_id"][i],
                    "title": batch["title"][i],
                    "text": batch["text"][i],
                }
            )
            for i , _ in enumerate(batch["_id"])
        ],
        batch_size=batch_size,
    )

1296it [16:48,  1.29it/s]                          


## embedding questions to ask for retreival after

In [ ]:
import tqdm
import numpy as np
import gc

# Initialize empty lists for embeddings
dense_vectors = []
sparse_vectors = []
late_vectors = []

# Process in batches using the proper dataset method
BATCH_SIZE = 16  # You can adjust this value

# Use the built-in iter method of HuggingFace datasets
for batch in tqdm.tqdm(dataset_questions.iter(batch_size=BATCH_SIZE), total=len(dataset_questions)//BATCH_SIZE):
    # batch["text"] is already a list of texts
    batch_texts = batch["text"]
    
    # Process in batch
    batch_dense_embeddings = list(dense_model.query_embed(batch_texts))
    batch_late_embeddings = list(late_interaction.query_embed(batch_texts))
    batch_sparse_embeddings = list(bm25_embedding_model.query_embed(batch_texts))
    
    # Add to main lists
    dense_vectors.extend(batch_dense_embeddings)
    late_vectors.extend(batch_late_embeddings)
    sparse_vectors.extend(batch_sparse_embeddings)
    
    # Optional: Clear some memory
    gc.collect()

70it [00:13,  5.22it/s]                        


In [ ]:
print(dense_vectors)

[array([-4.30211343e-02,  2.54887180e-02,  2.06948045e-02, -1.21698470e-02,
       -2.69618019e-02, -5.86789357e-02, -6.19167667e-02,  7.27071753e-02,
       -1.04004045e-02,  2.68960176e-02,  8.48984181e-03, -1.74287129e-02,
       -3.74542173e-02,  4.55348961e-02, -1.12349051e-02,  1.26510853e-02,
       -8.22896545e-02, -2.14705968e-02,  2.95099874e-02,  9.79199166e-02,
       -4.42724259e-02,  1.00978145e-02,  6.11675176e-02, -7.70360633e-03,
       -2.15976356e-02,  6.69642421e-02,  3.10534745e-02, -4.92939030e-02,
       -2.02597007e-02, -3.55476089e-02,  4.97432503e-02, -2.17783765e-02,
       -7.35356380e-02, -3.53165726e-02,  6.78260433e-02, -4.89405874e-02,
        5.41338135e-02,  4.72157719e-02, -4.32854223e-02,  3.99363184e-02,
        5.54177450e-02,  1.19082632e-02,  3.75087505e-02,  1.21026985e-02,
        8.76584368e-02, -6.14843180e-02,  1.91576850e-02, -9.75741517e-02,
       -2.99465216e-02, -1.13693317e-01,  1.72559256e-02,  1.44742443e-02,
       -1.31400318e-02, 

### code so i dont use ranx for evaluate / run / precision@k / mrr@k

In [46]:
class SimpleRun:
    def __init__(self, run_dict, name=None):
        self.run_dict = run_dict
        self.name = name
    
    def __str__(self):
        return f"SimpleRun(name={self.name}, queries={len(self.run_dict)})"

# Define evaluation metrics
def precision_at_k(qrels_dict, run_dict, k=10):
    """Calculate precision@k for all queries"""
    precisions = []
    
    for query_id, doc_scores in run_dict.items():
        if query_id not in qrels_dict:
            continue
            
        # Get top k documents
        top_k_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:k]
        top_k_doc_ids = [doc_id for doc_id, _ in top_k_docs]
        
        # Count relevant documents in top k
        relevant_docs = sum(1 for doc_id in top_k_doc_ids 
                           if doc_id in qrels_dict[query_id] and qrels_dict[query_id][doc_id] > 0)
        
        # Calculate precision
        precision = relevant_docs / k if k > 0 else 0
        precisions.append(precision)
    
    # Return average precision across all queries
    return sum(precisions) / len(precisions) if precisions else 0

def mrr_at_k(qrels_dict, run_dict, k=10):
    """Calculate Mean Reciprocal Rank@k for all queries"""
    reciprocal_ranks = []
    
    for query_id, doc_scores in run_dict.items():
        if query_id not in qrels_dict:
            continue
            
        # Get top k documents
        top_k_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:k]
        
        # Find first relevant document
        reciprocal_rank = 0
        for rank, (doc_id, _) in enumerate(top_k_docs, start=1):
            if doc_id in qrels_dict[query_id] and qrels_dict[query_id][doc_id] > 0:
                reciprocal_rank = 1.0 / rank
                break
                
        reciprocal_ranks.append(reciprocal_rank)
    
    # Return average MRR across all queries
    return sum(reciprocal_ranks) / len(reciprocal_ranks) if reciprocal_ranks else 0

# Custom evaluate function
def simple_evaluate(qrels_dict, run_dict, metrics=None):
    """Evaluate a run against qrels using specified metrics"""
    if metrics is None:
        metrics = ["precision@10", "mrr@10"]
    
    results = {}
    
    for metric in metrics:
        if metric == "precision@10":
            results[metric] = precision_at_k(qrels_dict, run_dict, k=10)
        elif metric == "mrr@10":
            results[metric] = mrr_at_k(qrels_dict, run_dict, k=10)
    
    return results

### for each embedding model
- dense_vectors

- sparse_vectors

- late_vectors

In [62]:
run_dic = {}

for idx , query in enumerate(dataset_questions):
    query_id = str(query["_id"])
    question_dense_vector = dense_vectors[idx]
    
    results = client.query_points(
        "scifact",
        query = question_dense_vector ,
        using = "all-MiniLM-L6-v2",
        with_payload=False,
        limit = 10,
    )
    run_dic[query_id] = {
        str(point.id): point.score
        for point in results.points
    }
dense_run = SimpleRun(run_dic, name="dense_run")

evaluation_results = simple_evaluate(
    custom_qrels.qrels,
    run_dic,
    metrics=["precision@10", "mrr@10"]
)
print(evaluation_results)

{'precision@10': 0.08529048207663782, 'mrr@10': 0.5842746286813036}


In [68]:
run_dic = {}

for idx , query in enumerate(dataset_questions):
    query_id = str(query["_id"])
    question_sparse_vector = sparse_vectors[idx]
    
    results = client.query_points(
        "scifact",
        query = models.SparseVector(
            indices = question_sparse_vector.as_object()["indices"].tolist(),
            values = question_sparse_vector.as_object()["values"].tolist()
        ) ,
        using = "bm25",
        with_payload=False,
        limit = 10,
    )
    run_dic[query_id] = {
        str(point.id): point.score
        for point in results.points
    }
dense_run = SimpleRun(run_dic, name="sparse_run")

evaluation_results = simple_evaluate(
    custom_qrels.qrels,
    run_dic,
    metrics=["precision@10", "mrr@10"]
)
print(evaluation_results)

{'precision@10': 0.0892459826946848, 'mrr@10': 0.6569917789942512}


In [69]:
run_dic = {}

for idx , query in enumerate(dataset_questions):
    query_id = str(query["_id"])
    question_late_vector = late_vectors[idx]
    
    results = client.query_points(
        "scifact",
        query = question_late_vector ,
        using = "colbertv2.0",
        with_payload=False,
        limit = 10,
    )
    run_dic[query_id] = {
        str(point.id): point.score
        for point in results.points
    }
dense_run = SimpleRun(run_dic, name="late_run")

evaluation_results = simple_evaluate(
    custom_qrels.qrels,
    run_dic,
    metrics=["precision@10", "mrr@10"]
)
print(evaluation_results)

{'precision@10': 0.09060568603213844, 'mrr@10': 0.6673004100692604}
